# Bronze validation

## Validate the current batch

Run all cells with `MODE = "batch"` after the Bronze DAGs finish. This read-only mode checks both booking snapshots, the named dealer export, and distinct Kafka event IDs. Set `seed_config` in `silver_settings.json` to the JSON used for seeding. This supplies the batch ID and expected counts. The default dealer export name uses the lowercase batch ID. Set `BATCH_DEALER_EXPORT_OVERRIDE` only for a custom export name. Blank batch REST and SQL IDs use the latest completed runs. Pin them to exact Airflow run IDs if needed.

## Compare Round 2 before and after

Place this notebook in `silver/` beside `bronze_review.py` and use the same VS Code kernel as the other Silver notebooks. The Round 2 baseline mode reads Bronze and saves a local checkpoint JSON file.

1. Run all four Bronze DAGs with sources unchanged. Verify the runs succeeded in Airflow and note the REST and SQL run IDs. Confirm the dealer files from the original export are already in Bronze.
2. Set `MODE = "baseline"`. Run all cells **before seeding**. Blank IDs select the latest complete Bronze snapshots and the latest dealer export. The notebook saves the selected manifests and the file and Kafka keys.
3. Apply the round-two seed, publish the dealer export once, and rerun all four DAGs. Verify success in Airflow. Set `MODE = "compare"` and run all cells again. Blank IDs select the latest complete snapshots and the latest dealer export published since the baseline.

To override auto-selection, use the Airflow run IDs exactly as displayed, including `manual__`. Use a new checkpoint filename for a separate test cycle. This notebook keeps the original baseline file once it is saved. The report validates Bronze evidence. Airflow task success is verified in Airflow.

In [ ]:
MODE = "batch"  # "baseline" and "compare" are for Round 2 only.
BATCH_REST_RUN = ""
BATCH_SQL_RUN = ""
BATCH_DEALER_EXPORT_OVERRIDE = ""

CHECKPOINT_FILE = "bronze_round2_baseline.json"

# Leave IDs blank for automatic selection from Bronze manifests.
BASELINE_REST_RUN = ""
BASELINE_SQL_RUN = ""
BASELINE_DEALER_EXPORT = ""

AFTER_REST_RUN = ""
AFTER_SQL_RUN = ""
AFTER_DEALER_EXPORT = ""

# Use the custom reference field configured by ERP_ORDER_REF_FIELD.
ERP_REFERENCE_FIELD = "custom_otc_seed_id"

In [ ]:
import base64
import hashlib
import json
from pathlib import Path, PurePosixPath
from urllib.parse import quote

import pandas as pd
from IPython.display import display

from bronze_review import decode_file, open_bronze

if MODE == "batch":
    import silver_notebook
    from silver_notebook import seed_config, settings_from_file

    settings_path = Path(silver_notebook.__file__).resolve().with_name("silver_settings.json")
    config = seed_config(settings_path, settings_from_file(settings_path))
    if config is None:
        raise ValueError("Set seed_config in silver_settings.json to the JSON used for seeding.")
    BATCH_ID = config["batch_id"].strip().upper()
    EXPECTED_PARENT_BOOKINGS = config["parent"]["bookings"]
    EXPECTED_DEALER_BOOKINGS = config["dealer"]["bookings"]
    storefront = config["storefront"]
    EXPECTED_KAFKA_EVENTS = (
        storefront["parent_booked_acceptances"] + 2 * storefront["parent_cancelled_orders"]
    )
    needs_dealer_export = EXPECTED_DEALER_BOOKINGS > 0 or config["dealer"]["pos_receipts"] > 0
    BATCH_DEALER_EXPORT = BATCH_DEALER_EXPORT_OVERRIDE.strip()
    if not BATCH_DEALER_EXPORT and needs_dealer_export:
        BATCH_DEALER_EXPORT = f"dealer-export-{BATCH_ID.lower()}"

bronze = open_bronze()
checkpoint_path = Path(CHECKPOINT_FILE)
dealer_prefix = "files/otc-inbound/dealer/exports/"
kafka_prefix = "kafka/parent.orders.v1/"


def snapshot_keys(rest_run, sql_run):
    return {
        source: f"batch/{source}/{quote(run_id, safe='')}/manifest.json"
        for source, run_id in (
            ("erpnext", rest_run), ("parent-pos", rest_run), ("dealer-sql", sql_run)
        )
    }


def object_keys():
    return {
        "file_manifests": sorted(key for key in bronze.objects
                                 if key.startswith("files/") and key.endswith("/manifest.json")),
        "kafka_objects": sorted(key for key in bronze.objects
                               if key.startswith(kafka_prefix) and key.endswith(".json")),
    }


def manifest(key):
    if key not in bronze.objects:
        raise ValueError(f"Bronze manifest missing: {key}")
    return json.loads(bronze.read(key))


def dealer_manifest(export_id, keys):
    prefix = f"{dealer_prefix}{export_id}/"
    matches = [key for key in keys if key.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one manifest under {prefix}; found {len(matches)}")
    return matches[0]


def collection_items(key, *, file_export=False):
    items = manifest(key)["files"]
    if file_export:
        return {PurePosixPath(item["key"]).stem.rsplit("-", 1)[0]: item for item in items}
    return {item["collection"]: item for item in items}


def checked_records(item, *, parent_manifest=None):
    key = item["key"]
    if parent_manifest is not None:
        key = f"{parent_manifest.rsplit('/', 1)[0]}/objects/{key}"
    body = bronze.read(key)
    if hashlib.sha256(body).hexdigest() != item["sha256"]:
        raise ValueError(f"SHA-256 mismatch: {key}")
    rows, _ = decode_file(key, body)
    if rows is None or len(rows) != item["rows"]:
        raise ValueError(f"Manifest row count mismatch: {key}")
    return rows


def latest_run(*sources):
    runs_by_source = []
    for source in sources:
        prefix = f"batch/{source}/"
        runs = {}
        for key in bronze.objects:
            if key.startswith(prefix) and key.endswith("/manifest.json"):
                data = manifest(key)
                runs[data["run_id"]] = pd.Timestamp(data["published_at"])
        runs_by_source.append(runs)
    complete = set.intersection(*(set(runs) for runs in runs_by_source))
    if not complete:
        raise ValueError(f"No complete published run for {', '.join(sources)}. Check the DAG tasks in Airflow.")
    return max(complete, key=lambda run: (max(runs[run] for runs in runs_by_source), run))


def latest_export(keys):
    keys = [key for key in keys if key.startswith(dealer_prefix)]
    if not keys:
        raise ValueError("No dealer export manifest found in the selected Bronze objects. Check the dealer export Job and Files DAG.")
    return max(keys, key=lambda key: (pd.Timestamp(manifest(key)["cutoff"]), key))


def acceptance_deliveries(keys):
    matching = []
    for key in keys:
        envelope = json.loads(bronze.read(key))
        encoded = envelope.get("value_base64")
        if encoded:
            event = json.loads(base64.b64decode(encoded))
            if isinstance(event, dict) and event.get("event_id") == "P-BRONZE-R2-EVENT":
                matching.append(event)
    return matching

def batch_report():
    batch = BATCH_ID.strip().upper()
    if not batch:
        raise ValueError("Set BATCH_ID to the ID in the seed config.")

    rest_run = BATCH_REST_RUN.strip() or latest_run("erpnext", "parent-pos")
    sql_run = BATCH_SQL_RUN.strip() or latest_run("dealer-sql")
    keys = snapshot_keys(rest_run, sql_run)
    rows = []

    def count(name, ids, expected, evidence, *, allow_replays=False):
        found = len(set(ids))
        if len(ids) != found and not allow_replays:
            status = "DUPLICATE IDS"
        elif expected is None:
            status = "COUNT ONLY" if found else "NO MATCHES"
        else:
            status = "PASS" if found == expected else "COUNT MISMATCH"
        rows.append({"check": name, "expected": expected, "found": found,
                     "status": status, "evidence": evidence})

    for source, collection, field, prefix, target in (
        ("erpnext", "Sales Order", ERP_REFERENCE_FIELD, f"P-{batch}-", EXPECTED_PARENT_BOOKINGS),
        ("dealer-sql", "sales_orders", "id", f"D-{batch}-", EXPECTED_DEALER_BOOKINGS),
    ):
        records = checked_records(collection_items(keys[source])[collection])
        identifiers = [record[field] for record in records if str(record.get(field, "")).startswith(prefix)]
        count(f"{source} bookings", identifiers, target, keys[source])

    captured = object_keys()
    if BATCH_DEALER_EXPORT.strip():
        export_prefix = f"{dealer_prefix}{BATCH_DEALER_EXPORT.strip()}/"
        exports = [key for key in captured["file_manifests"] if key.startswith(export_prefix)]
        count("Dealer export manifest", exports, 1, exports[0] if len(exports) == 1 else export_prefix)

    prefix = f"P-{batch}-"
    events = []
    for key in captured["kafka_objects"]:
        envelope = json.loads(bronze.read(key))
        encoded = envelope.get("value_base64")
        if not encoded:
            continue
        event = json.loads(base64.b64decode(encoded))
        event_id = event.get("event_id") if isinstance(event, dict) else None
        if isinstance(event_id, str) and event_id.startswith(prefix):
            events.append(event_id)
    count("Distinct parent Kafka events", events, EXPECTED_KAFKA_EVENTS,
          f"{len(events)} matching deliveries across Bronze Kafka objects", allow_replays=True)
    display(pd.DataFrame(rows).astype({"expected": "Int64", "found": "Int64"}))

In [ ]:
if MODE == "batch":
    batch_report()
elif MODE in {"baseline", "compare"}:
    report = []


    def add(dag, check, before, after, expected, passed):
        report.append({
            "DAG": dag, "Check": check, "Before": before, "After": after,
            "Expected change": expected, "Result": "PASS" if passed else "FAIL",
        })


    def compare_rows(dag, label, before, after, field, expected_ids, expected_delta):
        expected_ids = set(expected_ids)
        old_ids = {str(row.get(field, "")) for row in before} & expected_ids
        new_ids = {str(row.get(field, "")) for row in after} & expected_ids
        add(dag, label, f"{len(before)} rows; matches: {', '.join(sorted(old_ids)) or '0'}",
            f"{len(after)} rows; matches: {', '.join(sorted(new_ids)) or '0'}",
            f"Delta {expected_delta:+d}; new IDs: {', '.join(sorted(expected_ids))}",
            len(after) - len(before) == expected_delta and not old_ids and new_ids == expected_ids)


    if MODE == "baseline":
        if checkpoint_path.is_file():
            saved = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            print(f"Using the existing baseline in {checkpoint_path}.")
        else:
            rest_run = BASELINE_REST_RUN.strip() or latest_run("erpnext", "parent-pos")
            sql_run = BASELINE_SQL_RUN.strip() or latest_run("dealer-sql")
            keys = snapshot_keys(rest_run, sql_run)
            for key in keys.values():
                manifest(key)
            captured = object_keys()
            export_id = BASELINE_DEALER_EXPORT.strip()
            export_key = (dealer_manifest(export_id, captured["file_manifests"])
                          if export_id else latest_export(captured["file_manifests"]))

            parent_orders = checked_records(collection_items(keys["erpnext"])["Sales Order"])
            dealer_orders = checked_records(collection_items(keys["dealer-sql"])["sales_orders"])
            if (any(row.get(ERP_REFERENCE_FIELD) == "P-BRONZE-R2-ORDER" for row in parent_orders)
                or any(row.get("id") == "D-BRONZE-R2-ORDER" for row in dealer_orders)
                or acceptance_deliveries(captured["kafka_objects"])):
                raise ValueError("Round-two records are already visible in the selected Bronze evidence. Use the checkpoint captured before seeding.")

            saved = {"snapshots": keys, "dealer_export": export_key, **captured}
            with checkpoint_path.open("x", encoding="utf-8") as file:
                json.dump(saved, file, indent=2)

        display(pd.DataFrame([
            {"Source": source, "Baseline manifest": key}
            for source, key in {**saved["snapshots"], "dealer-export": saved["dealer_export"]}.items()
        ]))
        display(pd.DataFrame([
            {"Evidence": "Batch manifests", "Captured": len(saved["snapshots"])},
            {"Evidence": "Inbound manifests", "Captured": len(saved["file_manifests"])},
            {"Evidence": "Kafka objects", "Captured": len(saved["kafka_objects"])},
        ]))

    elif MODE == "compare":
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Capture the pre-seed baseline first: {checkpoint_path}")
        saved = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        rest_run = AFTER_REST_RUN.strip() or latest_run("erpnext", "parent-pos")
        sql_run = AFTER_SQL_RUN.strip() or latest_run("dealer-sql")
        after_keys = snapshot_keys(rest_run, sql_run)
        display(pd.DataFrame([
            {"Source": source, "Baseline manifest": saved["snapshots"][source], "After manifest": key}
            for source, key in after_keys.items()
        ]))
        if any(key == saved["snapshots"][source] for source, key in after_keys.items()):
            raise ValueError("Choose post-seed REST and SQL run IDs that differ from the baseline.")
        captured = object_keys()

        expected_batch = {
            "erpnext": [
                ("Sales Order", ERP_REFERENCE_FIELD, ["P-BRONZE-R2-ORDER"], 1),
                ("Stock Entry", ERP_REFERENCE_FIELD, ["P-BRONZE-R2-STOCK"], 1),
                ("Delivery Note", ERP_REFERENCE_FIELD, ["P-BRONZE-R2-SHIP"], 1),
                ("Sales Invoice", ERP_REFERENCE_FIELD, ["P-BRONZE-R2-INVOICE", "P-BRONZE-R2-CREDIT"], 2),
            ],
            "parent-pos": [("pos_receipts", "receipt_id", ["P-BRONZE-R2-POS"], 1)],
            "dealer-sql": [
                ("sales_orders", "id", ["D-BRONZE-R2-ORDER"], 1),
                ("order_lines", "id", ["D-BRONZE-R2-LINE"], 1),
                ("shipments", "id", ["D-BRONZE-R2-SHIP"], 1),
            ],
        }
        for source, expectations in expected_batch.items():
            dag = "SQL" if source == "dealer-sql" else "REST"
            before = collection_items(saved["snapshots"][source])
            after = collection_items(after_keys[source])
            for collection, field, ids, delta in expectations:
                compare_rows(dag, f"{source}/{collection}",
                             checked_records(before[collection]), checked_records(after[collection]),
                             field, ids, delta)
            if source == "erpnext":
                before_gl = checked_records(before["GL Entry"])
                after_gl = checked_records(after["GL Entry"])
                invoices = checked_records(after["Sales Invoice"])
                for ref in ("P-BRONZE-R2-INVOICE", "P-BRONZE-R2-CREDIT"):
                    vouchers = {row["name"] for row in invoices if row.get(ERP_REFERENCE_FIELD) == ref}
                    old = sum(row.get("voucher_no") in vouchers for row in before_gl)
                    new = sum(row.get("voucher_no") in vouchers for row in after_gl)
                    add(dag, f"GL lines linked to {ref}", old, new, "New voucher postings", len(vouchers) == 1 and old == 0 and new > 0)
            if source == "dealer-sql":
                for collection in ("customers", "products", "warehouses", "accounts"):
                    old, new = before[collection], after[collection]
                    checked_records(old)
                    checked_records(new)
                    add(dag, f"Unchanged {collection}", old["rows"], new["rows"],
                        "Same row count and SHA-256",
                        (old["rows"], old["sha256"]) == (new["rows"], new["sha256"]))

        new_files = set(captured["file_manifests"]) - set(saved["file_manifests"])
        new_dealer = [key for key in new_files if key.startswith(dealer_prefix)]
        export_id = AFTER_DEALER_EXPORT.strip()
        if export_id:
            target = [key for key in new_dealer if key.startswith(f"{dealer_prefix}{export_id}/")]
        else:
            target = [latest_export(new_dealer)] if new_dealer else []
        add("Files", "New dealer export manifest", 0, len(new_dealer),
            "Exactly one new dealer export" + (f" for {export_id}" if export_id else ""),
            len(new_dealer) == len(target) == 1)
        file_expectations = [
            ("invoices", ["D-BRONZE-R2-INVOICE", "D-BRONZE-R2-CREDIT"], 2),
            ("receivables", ["D-BRONZE-R2-AR"], 1),
            ("gl_entries", [f"D-BRONZE-R2-GL{i}" for i in range(1, 5)], 4),
        ]
        if len(target) == 1:
            old_key, new_key = saved["dealer_export"], target[0]
            old_files = collection_items(old_key, file_export=True)
            new_payloads = collection_items(new_key, file_export=True)
            for table, ids, delta in file_expectations:
                compare_rows("Files", f"Dealer CSV/{table}",
                             checked_records(old_files[table], parent_manifest=old_key),
                             checked_records(new_payloads[table], parent_manifest=new_key),
                             "id", ids, delta)

        baseline_events = set(saved["kafka_objects"])
        old_deliveries = acceptance_deliveries(sorted(baseline_events))
        new_deliveries = acceptance_deliveries(sorted(set(captured["kafka_objects"]) - baseline_events))
        valid = all(event.get("order_id") == "P-BRONZE-R2-ORDER" and event.get("status") == "accepted"
                    for event in new_deliveries)
        add("Kafka", "P-BRONZE-R2-EVENT deliveries", len(old_deliveries), len(new_deliveries),
            "Absent before; at least one delivery of the seeded acceptance after",
            not old_deliveries and bool(new_deliveries) and valid)

        comparison_df = pd.DataFrame(report)
        display(comparison_df)
        print(comparison_df["Result"].value_counts().to_string())

else:
    raise ValueError('MODE must be "batch", "baseline", or "compare".')